# 02 - Análisis Exploratorio de Datos (EDA)

**Proyecto:** Estacionalidad y composición del delito en la Región de Valparaíso (EIN092B, Avance 2)

**Pregunta:** ¿Qué familias y tipos de delito predominan en la Región de Valparaíso y cómo evoluciona su frecuencia mes a mes y año a año (2019-2025)?

- **X:** familia de delito, tipo de delito, mes, año
- **Y:** casos policiales
- **T:** 2019-2026, mensual (2026 parcial)

**Fuente:** CEAD (Casos Policiales), Región de Valparaíso.

Flujo: Datos -> EDA -> Visualización -> Hallazgos. Cada gráfico va seguido de su interpretación.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
TEAL, CLAY = '#2C6E75', '#B25B3E'

def short(s):
    return (str(s).replace('Delitos contra la ','').replace('Delitos ','')
            .replace('vida o integridad de las personas','Vida e integridad')
            .replace('propiedad no violentos','Propiedad no violentos')
            .replace('asociados a ','').capitalize())

df = pd.read_csv('../data/processed/delitos_valparaiso_mensual.csv')
fam = df[df['nivel'] == 'familia'].copy()
sub = df[df['nivel'] == 'subtipo'].copy()
df.head()

## 1. Estructura y calidad de los datos

Revisamos tamaño, tipos, faltantes, duplicados, rangos y cobertura antes de analizar.

In [ ]:
print('Observaciones:', len(df))
print('Variables   :', df.shape[1])
print('\nTipos:'); print(df.dtypes)
print('\nFaltantes por columna:'); print(df.isna().sum())
print('\nFilas duplicadas:', df.duplicated().sum())
print('\nRangos: anio', df.anio.min(), '-', df.anio.max(),
      '| mes', df.mes.min(), '-', df.mes.max(),
      '| casos', df.casos.min(), '-', df.casos.max())

In [ ]:
# Cobertura temporal: meses con datos por año
cobertura = fam.groupby('anio')['mes'].nunique()
print('Meses con datos por año:'); print(cobertura)

# Consistencia jerárquica: en las familias con subtipos, el total = suma de sus subtipos
fams_con_sub = sorted(sub['familia'].unique())
ft = fam[fam.familia.isin(fams_con_sub)].groupby('familia')['casos'].sum()
st = sub.groupby('familia')['casos'].sum()
print('\nFamilias con subtipos :', len(fams_con_sub))
print('Familias sin subtipos :', sorted(set(fam.familia.unique()) - set(fams_con_sub)))
print('Máx diferencia (familia vs suma de sus subtipos):', (ft - st).abs().max())

**Interpretación (calidad y decisiones tomadas):**

- **Sin faltantes ni duplicados**: `data_prep.py` ya descartó los meses futuros vacíos y normalizó el formato numérico chileno (miles con punto) del CEAD.
- **Cobertura temporal**: 2019-2025 completos (12 meses); **2026 es parcial** (solo hasta junio). Decisión: se excluye 2026 de las comparaciones anuales y se marca cuando aparece.
- **Estructura jerárquica**: 5 de las 8 familias traen desglose de subtipos, y en ellas el total coincide **exactamente** con la suma de sus subtipos (diferencia 0). Tres familias (drogas, violencia intrafamiliar, otros delitos o faltas) no traen subtipos en el export. Decisión: NO se suman familias + subtipos juntos (evita doble conteo); el análisis principal usa `nivel == 'familia'`.
- **Cobertura espacial**: el dato es regional agregado (sin desglose por comuna). Se declara como límite del proyecto.
- **Unidad de observación**: casos policiales de un tipo de delito en un mes y año dados.

## 2. Análisis univariado

Distribución de la variable objetivo (casos por familia) y frecuencia de las categorías.

In [ ]:
print(fam['casos'].describe())
plt.figure(figsize=(9,3.5))
sns.histplot(fam['casos'], kde=True, color=TEAL)
plt.title('Distribución de casos mensuales por familia'); plt.xlabel('Casos'); plt.tight_layout(); plt.show()

In [ ]:
comp = fam[fam.anio == 2025].groupby('familia')['casos'].sum().sort_values()
comp_pct = 100 * comp / comp.sum()
plt.figure(figsize=(9,4))
colors = [CLAY if i == len(comp)-1 else TEAL for i in range(len(comp))]
plt.barh([short(x) for x in comp.index], comp.values, color=colors)
plt.title('Composición del delito por familia (2025)'); plt.xlabel('Casos'); plt.tight_layout(); plt.show()
print(comp_pct.round(1).sort_values(ascending=False))

**Interpretación:**

- La distribución de casos es **muy asimétrica**: unas pocas familias concentran la mayor parte del volumen.
- En 2025, **"Otros delitos o faltas" concentra ~42%** de los casos, seguido de **propiedad no violentos (~17%)** e **incivilidades (~16%)**. Las familias violentas (robos violentos, drogas, armas) pesan mucho menos en volumen.
- Esto orienta el foco: la carga del delito en Valparaíso está dominada por faltas, delitos contra la propiedad e incivilidades, no por delitos violentos.

## 3. Relaciones entre variables

¿Cómo se relaciona Y (casos) con las categorías (familia) y el tiempo (mes)? Comportamiento conjunto de dos dimensiones.

In [ ]:
base = fam[fam.anio <= 2025].copy()
piv = base.pivot_table(index='familia', columns='mes', values='casos', aggfunc='mean')
piv.index = [short(x) for x in piv.index]
plt.figure(figsize=(11,4.5))
sns.heatmap(piv, cmap='rocket_r', cbar_kws={'label':'Casos promedio'})
plt.title('Casos promedio por familia y mes (2019-2025)')
plt.xticks(range(12), MESES); plt.xlabel('Mes'); plt.ylabel(''); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,4.5))
order = base.groupby('familia')['casos'].median().sort_values().index
sns.boxplot(data=base, x='casos', y='familia', order=order,
            palette='crest', showfliers=True)
plt.yticks(range(len(order)), [short(x) for x in order])
plt.title('Dispersión de casos mensuales por familia (2019-2025)'); plt.xlabel('Casos'); plt.ylabel(''); plt.tight_layout(); plt.show()

**Interpretación:**

- El heatmap muestra que la **magnitud depende sobre todo de la familia** (filas), y dentro de cada familia hay variación mensual moderada (columnas): hay señal estacional pero no domina sobre la diferencia entre familias.
- El boxplot confirma la asimetría: "Otros delitos o faltas" tiene mediana y dispersión mucho mayores; las familias violentas son compactas y bajas.
- Los valores extremos (fliers) coinciden en su mayoría con **2020-2021** (caída por pandemia), no con errores de datos.

## 4. Componente temporal

Evolución mes a mes, tendencia interanual y estacionalidad (el núcleo de la pregunta).

In [ ]:
fam['fecha'] = pd.to_datetime(dict(year=fam.anio, month=fam.mes, day=1))
figp = px.line(fam, x='fecha', y='casos', color='familia',
               title='Casos mensuales por familia - Valparaíso (2019-2026)')
figp.update_layout(xaxis_title='Mes', yaxis_title='Casos', legend_title='Familia')
figp.show()

In [ ]:
anual = fam[fam.anio <= 2025].groupby(['anio','familia'], as_index=False)['casos'].sum()
figa = px.line(anual, x='anio', y='casos', color='familia', markers=True,
               title='Tendencia interanual por familia (2019-2025)')
figa.update_layout(xaxis_title='Año', yaxis_title='Casos', legend_title='Familia')
figa.show()

In [ ]:
objetivo = 'Delitos contra la propiedad no violentos'
b = fam[(fam.familia == objetivo) & (fam.anio <= 2025)]
prom = b.groupby('mes')['casos'].mean()
indice = 100 * prom / prom.mean()
plt.figure(figsize=(10,3.8))
sns.barplot(x=list(range(1,13)), y=indice.values,
            palette=[CLAY if v>=100 else TEAL for v in indice.values])
plt.axhline(100, color='gray', ls='--', lw=1)
plt.xticks(range(12), MESES)
plt.title(f'Índice estacional - {objetivo} (100 = mes promedio)'); plt.ylabel('Índice'); plt.tight_layout(); plt.show()
print(indice.round(0))

**Interpretación:**

- **Tendencia**: caída marcada en **2020-2021** (pandemia) y recuperación posterior. "Otros delitos o faltas" crece de forma sostenida hacia 2024-2025; **las incivilidades caen fuerte** (de ~60.847 en 2019 a ~30.473 en 2023).
- **Estacionalidad**: los **delitos contra la propiedad suben en verano** (enero ~+14% sobre el promedio) y bajan a mitad de año (junio ~-8%). Es un patrón consistente año a año.
- Ambos hallazgos responden directamente a la pregunta: **qué predomina** (composición) y **cuándo** (estacionalidad).

## 5. Hallazgos preliminares

De "estos son nuestros datos" a **"esto es lo que hemos aprendido"**:

1. La carga del delito está **dominada por faltas, propiedad e incivilidades**, no por delitos violentos (composición muy asimétrica).
2. Hay una **caída pandémica clara (2020-2021)** común a casi todas las familias.
3. Las **incivilidades bajan de forma sostenida** desde 2019; "otros delitos o faltas" es lo que más crece.
4. Los **delitos contra la propiedad tienen estacionalidad de verano** (peak en enero).

**Cómo refina la pregunta:** el foco más prometedor es cruzar **composición x estacionalidad** por familia, y explicar la caída/recuperación 2020-2025. La dimensión espacial (comuna) queda fuera por los datos disponibles.

Estas vistas se llevan a la app Streamlit (páginas Contexto / EDA / Análisis de la pregunta).